# 02 - Resolve ENA accessions and build a retrieval plan

Notebook 01 created an isolate-level phenotype manifest. The published supplementary table mixes direct run accessions with sample and experiment identifiers. This notebook resolves those public identifiers through ENA and creates an inspectable FASTQ retrieval plan without downloading sequence files.

## 1. Configuration

Resolution metadata is cached under `data/raw/ena_metadata/`. Rerunning the notebook therefore avoids repeated ENA requests.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from disinfectant_tolerance.ena import (
    extract_accessions,
    https_urls,
    load_cache,
    resolve_accession_cell,
    save_cache,
    sum_semicolon_ints,
)

MANIFEST_PATH = PROJECT_ROOT / "data" / "manifests" / "isolate_manifest.csv"
CACHE_PATH = PROJECT_ROOT / "data" / "raw" / "ena_metadata" / "read_run_cache.json"
OUTPUT_DIR = PROJECT_ROOT / "data" / "manifests"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

## 2. Inspect accession types

The feasibility cohort should contain 200 isolates. This inspection makes mixed archive identifiers visible before any API calls are made.

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH, dtype=str).fillna("")
manifest["accession_tokens"] = manifest["ena_accession"].map(extract_accessions)
manifest["accession_prefix"] = manifest["accession_tokens"].map(
    lambda values: values[0][:3] if values else "missing"
)

print("Isolates:", len(manifest))
print("\nLabels:")
print(manifest["label"].value_counts().sort_index())
print("\nFirst accession prefixes:")
print(manifest["accession_prefix"].value_counts())
manifest[["isolate_id", "ena_accession", "accession_tokens"]].head(12)

## 3. Resolve public read runs

Each isolate remains one row in the resolved manifest. A separate run-level table records downloadable FASTQ URLs and byte estimates. Failed lookups are retained in the audit table rather than silently discarded.

In [ ]:
cache = load_cache(CACHE_PATH)
resolved_rows = []
retrieval_rows = []
audit_rows = []

for position, row in manifest.iterrows():
    isolate_id = row["isolate_id"]
    accession_cell = row["ena_accession"]
    error = ""
    try:
        runs = resolve_accession_cell(accession_cell, cache=cache)
    except Exception as exc:
        runs = []
        error = f"{type(exc).__name__}: {exc}"

    run_accessions = []
    total_bytes = 0
    for run in runs:
        run_accession = run.get("run_accession", "")
        urls = https_urls(run.get("fastq_ftp", ""))
        fastq_bytes = sum_semicolon_ints(run.get("fastq_bytes", ""))
        run_accessions.append(run_accession)
        total_bytes += fastq_bytes
        retrieval_rows.append({
            "isolate_id": isolate_id,
            "label": row["label"],
            "group": row["group"],
            "source_accession": accession_cell,
            "run_accession": run_accession,
            "sample_accession": run.get("sample_accession", ""),
            "secondary_sample_accession": run.get("secondary_sample_accession", ""),
            "study_accession": run.get("study_accession", ""),
            "scientific_name": run.get("scientific_name", ""),
            "fastq_urls": ";".join(urls),
            "fastq_bytes": fastq_bytes,
        })

    resolution_status = "resolved" if run_accessions else "unresolved"
    resolved = row.drop(labels=["accession_tokens", "accession_prefix"]).to_dict()
    resolved.update({
        "resolved_run_accessions": ";".join(run_accessions),
        "fastq_bytes": total_bytes,
        "fastq_gb": round(total_bytes / 1_000_000_000, 3),
        "resolution_status": resolution_status,
    })
    resolved_rows.append(resolved)
    audit_rows.append({
        "isolate_id": isolate_id,
        "source_accession": accession_cell,
        "resolution_status": resolution_status,
        "resolved_run_count": len(run_accessions),
        "error": error,
    })

    if (position + 1) % 20 == 0 or position + 1 == len(manifest):
        save_cache(CACHE_PATH, cache)
        print(f"Resolved {position + 1}/{len(manifest)} isolates")

resolved_manifest = pd.DataFrame(resolved_rows)
retrieval_plan = pd.DataFrame(retrieval_rows)
resolution_audit = pd.DataFrame(audit_rows)

## 4. Review and export

A few unresolved identifiers are acceptable at this stage, but they should be reviewed before scaling the experiment. The retrieval plan is metadata only.

In [ ]:
resolved_path = OUTPUT_DIR / "isolate_manifest_resolved.csv"
retrieval_path = OUTPUT_DIR / "ena_run_retrieval_plan.csv"
audit_path = OUTPUT_DIR / "ena_resolution_audit.csv"

resolved_manifest.to_csv(resolved_path, index=False)
retrieval_plan.to_csv(retrieval_path, index=False)
resolution_audit.to_csv(audit_path, index=False)

print("Resolution status:")
print(resolved_manifest["resolution_status"].value_counts(dropna=False))
print("\nResolved runs:", len(retrieval_plan))
print("Estimated FASTQ download: %.2f GB" % (retrieval_plan["fastq_bytes"].sum() / 1_000_000_000))
print("\nWrote:", resolved_path.relative_to(PROJECT_ROOT))
print("Wrote:", retrieval_path.relative_to(PROJECT_ROOT))
print("Wrote:", audit_path.relative_to(PROJECT_ROOT))

resolution_audit[resolution_audit["resolution_status"] != "resolved"].head(20)

## 5. Next step

Open `03_make_lineage_splits_and_smoke_plan.ipynb`. It creates lineage-aware train/dev/test splits and a small FASTQ acquisition plan. Do not download the full cohort yet.